# Malmo environment wrapper — example usage (Docker-friendly)

This notebook demonstrates the **standardized interface** (`reset`, `step`) and integration with a minimal **training harness**: create env, run episodes, loop over steps with a simple policy (here, random actions).

**Before running**: Minecraft with the Malmo mod must be running in noVNC (`http://127.0.0.1:6901`). The first `env.reset()` starts a mission and connects to Minecraft on port 10000.

## 1. Setup path and imports

In Docker the project is mounted at `/workspace`. We add that (or the project root) to `sys.path` so `env` can be imported.

In [ ]:
import os
import random
import sys

# Project root: Docker mounts repo at /workspace; otherwise use cwd.
if os.path.isdir("/workspace") and os.path.isfile("/workspace/env/mission.xml"):
    workspace = "/workspace"
else:
    workspace = os.getcwd()
    if not os.path.isfile(os.path.join(workspace, "env", "mission.xml")):
        workspace = os.path.dirname(workspace)
        if not os.path.isfile(os.path.join(workspace, "env", "mission.xml")):
            raise FileNotFoundError("Could not find env/mission.xml; set workspace to project root.")

sys.path.insert(0, workspace)
mission_path = os.path.join(workspace, "env", "mission.xml")
print("Workspace: {}".format(workspace))
print("Mission XML: {}".format(mission_path))

In [ ]:
from env.malmo_env import MalmoGridEnv

## 2. Observation and action space

- **Observation**: dict with `x`, `z`, `y` (agent position).
- **Actions**: 0=North, 1=South, 2=West, 3=East.
- **Termination** (in `info["termination_reason"]`): `success_diamond_picked_up`, `failure_fell_off_platform`, `timeout_max_steps_reached`, or `malmo_mission_ended_early`.

Policies you can use in the loop below:
- **Random**: `action = random.randint(0, 3)`
- **Go straight**: always the same direction, e.g. `action = 0` (North); use 0, 1, 2, or 3 for N, S, W, E.

## 3. Create environment and run episodes (training harness)

Minimal loop: **reset** → repeat **step** until **done** → log. This is the pattern any agent (random, Q-learning, etc.) will use.

In [ ]:
random.seed(42)
max_steps = 200
num_episodes = 2

# Policy: "random" or "straight". For straight, 0=N, 1=S, 2=W, 3=E.
policy = "straight"
straight_action = 0  # North

env = MalmoGridEnv(mission_xml_path=mission_path, max_steps=max_steps)

for episode in range(num_episodes):
    obs = env.reset(seed=42)
    done = False
    total_reward = 0.0
    step = 0
    while not done and step < max_steps:
        if policy == "straight":
            action = straight_action
        else:
            action = random.randint(0, 3)
        obs, reward, done, info = env.step(action)
        total_reward += reward
        step += 1
    reason = info.get("termination_reason", "?")
    print("Episode {} | steps={} | total_reward={} | reason={}".format(episode, step, total_reward, reason))

## 4. Debugging connection

- **"Nothing is listening on port 10000"** — Normal until a mission starts. Run the cell above; the first `reset()` starts the mission and connects.
- **Timeout waiting for mission to begin** — Ensure Minecraft is running in noVNC; restart the container if needed.
- **Import error for `env`** — Run the first cell so `workspace` is on `sys.path`.